# Multi-Instance GPU (MIG)

A practical reference for **Multi-Instance GPU (MIG)** — the NVIDIA hardware
feature that partitions a single data-center GPU (A100, A30, H100, H200, GH200,
B200, …) into up to **seven fully isolated GPU instances**. Each instance gets
its own dedicated streaming multiprocessors (SMs), L2 cache slices, memory, and
memory bandwidth, so workloads on one slice cannot interfere with — or crash —
workloads on another.

This notebook covers the MIG concepts (slices, GPU Instances, Compute
Instances, profiles), the raw `nvidia-smi mig` workflow used to create and tear
down partitions, how MIG devices surface to CUDA and Kubernetes, and the
operational gotchas. For the *declarative/fleet* tooling that drives this at
scale, see the companion **MIG Manager** notebook.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

### What is it?

**Multi-Instance GPU (MIG)** is a partitioning capability of NVIDIA Ampere and
newer data-center GPUs. It carves the physical GPU into as many as **7
independent GPU instances**, each behaving like a smaller, standalone GPU with a
**guaranteed, hardware-enforced** share of compute and memory.

The GPU is internally divided into fixed building blocks:

- **Memory slices** — an A100 has **8** memory slices (the H100 likewise exposes
  8). Each slice is a fixed fraction of framebuffer plus its share of the memory
  controllers and bandwidth.
- **Compute (SM) slices** — **7** usable SM slices (one slot is held back for
  management, which is why you get *7* instances, not 8).

A **GPU Instance (GI)** bundles some memory slices + some compute slices behind
its own fault-isolation boundary. A **Compute Instance (CI)** then subdivides the
SMs *inside* a GI. Profiles are named `<compute>g.<memory>gb` — e.g. `3g.20gb`
means 3 compute slices and 20 GB of memory.

### Why use it?

- **Hard isolation, not best-effort sharing.** Each instance has dedicated SMs,
  L2, DRAM and bandwidth. A noisy neighbor cannot steal cycles or bandwidth, and
  a fault/XID/OOM in one instance does not take down the others.
- **Predictable QoS.** Latency-sensitive inference gets deterministic
  performance because nothing else contends for its slice.
- **Higher utilization.** A full A100/H100 is overkill for small models,
  notebooks, or CI jobs. MIG packs many such jobs onto one card instead of
  stranding 80%+ of it.
- **Right-sizing.** Profiles let you match the instance to the workload's real
  compute *and* memory footprint.

### When to use it?

- You serve **many small/medium workloads** (inference endpoints, Jupyter users,
  CI runs) that each need *a* GPU, not a whole one.
- You need **multi-tenant isolation** with QoS guarantees stronger than
  time-slicing or MPS can give.
- You have **MIG-capable hardware** (A100/A30/H100/H200/GH200/B200). Consumer
  cards and V100/T4-class GPUs do **not** support MIG.

## Key Features

### Core capabilities of MIG

| Capability | Description | Why it matters |
|---|---|---|
| Hardware-partitioned instances | Up to 7 GPU Instances, each with dedicated SMs, L2, DRAM | True isolation, not time-multiplexing |
| Memory-bandwidth isolation | Each instance owns its memory controllers/bandwidth | No noisy-neighbor bandwidth contention |
| Fault & error isolation | An XID/uncorrectable error is contained to one instance | One tenant's crash can't kill the rest |
| Fixed profiles | `1g`, `2g`, `3g`, `4g`, `7g` compute × matching memory | Predictable, schedulable building blocks |
| Two-level slicing (GI → CI) | A GI can be further split into Compute Instances | Share an instance's memory while splitting its SMs |
| Per-instance scheduling/QoS | Each instance has its own engines & scheduler | Deterministic latency per tenant |
| CUDA-transparent | Each MIG device appears as a normal CUDA device via UUID | Apps need no code changes |
| Cloud-native integration | Advertised to Kubernetes via the device plugin + GFD | `nvidia.com/mig-<profile>` resources |

## Architecture Overview

```text
                 Physical A100/H100 GPU (MIG mode ENABLED)
   +---------------------------------------------------------------+
   |  8 memory slices        |   7 usable compute (SM) slices      |
   |  [m][m][m][m][m][m][m][m]|   [c][c][c][c][c][c][c]             |
   +---------------------------------------------------------------+
                 |  carve into GPU Instances (GIs)  |
                 v                                    v
   +-------------------------+   +-----------------+   +-----------+
   | GPU Instance  3g.20gb   |   | GI 2g.10gb      |   | GI 2g.10gb|
   |  3 SM + 3 mem slices    |   | 2 SM + 2 mem    |   | 2 SM+2mem |
   |  (own L2 + bandwidth)   |   |                 |   |           |
   |  +-------------------+  |   |  +-----------+  |   |  +------+ |
   |  | Compute Instance  |  |   |  | CI (full) |  |   |  | CI   | |
   |  | (split SMs here)  |  |   |  +-----------+  |   |  +------+ |
   |  +-------------------+  |   +-----------------+   +-----------+
   +-------------------------+
        |                          each GI -> one MIG-<UUID> CUDA device
        v
   CUDA_VISIBLE_DEVICES=MIG-<uuid>   ->   app sees an isolated "GPU"
```

### Components

1. **Memory slice** — smallest unit of framebuffer + memory bandwidth (1/8 of
   the GPU on A100/H100). Determines the `.NNgb` part of a profile.
2. **Compute (SM) slice** — a fixed group of SMs (1/7 of usable compute).
   Determines the `Ng` part of a profile.
3. **GPU Instance (GI)** — memory slices + compute slices behind a fault-isolation
   boundary. This is the isolation unit; created from a **GI profile**.
4. **Compute Instance (CI)** — a partition of a GI's SMs. A GI always has at
   least one CI (the default spanning all its SMs); you can split it further so
   several CIs share the GI's memory but get separate compute.
5. **MIG device** — what CUDA sees: a `(GI, CI)` pair exposed as `MIG-<UUID>`,
   selectable via `CUDA_VISIBLE_DEVICES`.

## Installation

MIG is **not software you install** — it is a mode you enable on supported
hardware. The "installation" is really: confirm the GPU supports MIG, ensure a
recent data-center driver, and turn MIG mode on.

### Prerequisites

- A **MIG-capable GPU**: A100, A30, H100, H200, GH200, B200, etc.
  (V100, T4, and consumer GeForce/RTX cards do **not** support MIG).
- **NVIDIA data-center driver R450+** (MIG has been supported since R450; use a
  current branch for H100/B200).
- **CUDA 11.0+** for application-side MIG awareness.
- **Root / privileged access** — enabling MIG mode and editing partitions is a
  privileged operation.
- The GPU must be **idle** (no CUDA clients attached) to toggle MIG mode; on some
  configurations a **GPU reset** is required for the mode change to take effect.

> For configuring MIG **declaratively across a cluster**, use `nvidia-mig-parted`
> and the GPU Operator's `k8s-mig-manager` (see the MIG Manager notebook). The
> cells here show the raw, single-host `nvidia-smi mig` workflow so you
> understand what that tooling automates.

In [ ]:
# --- Step 0: confirm the GPU supports MIG and check its current mode ---------
# Run these on the GPU host (they need a real NVIDIA data-center GPU + driver).

# List GPUs and current MIG mode:
#   nvidia-smi -L
#   nvidia-smi --query-gpu=index,name,mig.mode.current,mig.mode.pending --format=csv

# --- Step 1: enable MIG mode on GPU 0 (privileged; GPU must be idle) ---------
#   sudo nvidia-smi -i 0 -mig 1
# If it reports a pending change, free the GPU and reset it:
#   sudo nvidia-smi --gpu-reset -i 0      # or reboot the node
# Confirm:
#   nvidia-smi --query-gpu=index,mig.mode.current --format=csv

import shutil, subprocess

if shutil.which("nvidia-smi"):
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,mig.mode.current", "--format=csv"],
        capture_output=True, text=True,
    )
    print(out.stdout or out.stderr)
else:
    print("nvidia-smi not found here (no GPU in this environment).")
    print("On a MIG-capable host you would see, after `nvidia-smi -i 0 -mig 1`:")
    print("index, name, mig.mode.current")
    print("0, NVIDIA A100-SXM4-40GB, Enabled")

## Basic Usage

The single-host workflow has four moves:

1. **Enable MIG mode** on the GPU (previous section).
2. **List the available GPU-Instance profiles** for this exact GPU.
3. **Create GPU Instances** (and a default Compute Instance) from those profiles.
4. **Find the resulting MIG device UUIDs** and point CUDA at one.

Profiles are listed by `nvidia-smi mig -lgip`. Each row shows a **profile ID**, a
**name** (e.g. `1g.5gb`), how many **instances** of it can exist, and how many
memory/SM slices it consumes. You create instances either by **profile ID** or
by **name**.

In [ ]:
# --- A100-40GB GPU Instance profiles (output of `nvidia-smi mig -lgip`) ------
# Profile IDs are GPU-model specific; these are the canonical A100-40GB ones.
a100_40gb_profiles = '''
Profile        ID   Instances   Memory   SM slices
1g.5gb         19   7           5 GB     1
1g.5gb+me      20   1           5 GB     1  (with media engines)
2g.10gb        14   3           10 GB    2
3g.20gb         9   2           20 GB    3
4g.20gb         5   1           20 GB    4
7g.40gb         0   1           40 GB    7
'''
print(a100_40gb_profiles)

# --- Create instances ---------------------------------------------------------
# Create one 3g.20gb and two 2g.10gb... wait, geometry must fit (3+2+2 = 7 SMs):
#   sudo nvidia-smi mig -i 0 -cgi 3g.20gb,2g.10gb,2g.10gb -C
#
#   -cgi  : create GPU Instances (by name OR profile id, e.g. 9,14,14)
#   -C    : also create the DEFAULT Compute Instance in each new GI
#
# Or the classic "7 tiny slices" layout:
#   sudo nvidia-smi mig -i 0 -cgi 1g.5gb,1g.5gb,1g.5gb,1g.5gb,1g.5gb,1g.5gb,1g.5gb -C
#
# Inspect what now exists:
#   nvidia-smi mig -lgi          # list GPU Instances
#   nvidia-smi mig -lci          # list Compute Instances
#   nvidia-smi -L                # MIG devices with their UUIDs

print("After creation, `nvidia-smi -L` shows lines like:")
print("  GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-abc...)")
print("    MIG 3g.20gb Device 0: (UUID: MIG-1d4f...)")
print("    MIG 2g.10gb Device 1: (UUID: MIG-9a2c...)")
print("    MIG 2g.10gb Device 2: (UUID: MIG-77be...)")

## Advanced Features

#### Compute Instances: splitting a GI's SMs

A GPU Instance starts with one **default Compute Instance** spanning all its SMs.
You can instead split those SMs into several CIs that **share the GI's memory**
but get isolated compute — useful when several processes need the same data
footprint but separate, guaranteed compute shares.

List a GI's compute-instance profiles, then create them:

```bash
nvidia-smi mig -lcip -gi 1                 # CI profiles available in GI 1
sudo nvidia-smi mig -cci 1c.3g.20gb -gi 1  # carve CIs inside GI 1
```

#### Targeting a MIG device from CUDA

Each MIG device is a `(GI, CI)` pair exposed as `MIG-<UUID>`. Applications select
one with `CUDA_VISIBLE_DEVICES` — **using the MIG UUID, not an integer index**
(integer indexing is unsupported across MIG devices):

```bash
CUDA_VISIBLE_DEVICES=MIG-9a2c... python infer.py
```

#### Tearing partitions down

Destroy in reverse order (CIs first, then GIs); MIG mode can then be disabled:

```bash
sudo nvidia-smi mig -dci -gi 1     # destroy compute instances in GI 1
sudo nvidia-smi mig -dgi           # destroy all GPU instances
sudo nvidia-smi -i 0 -mig 0        # disable MIG mode (GPU must be idle)
```

In [ ]:
# Discover MIG devices the way an application would, and pick one for CUDA.
import shutil, subprocess

def list_mig_devices():
    """Return MIG device UUIDs visible on this host (empty if none/no GPU)."""
    if not shutil.which("nvidia-smi"):
        return []
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader", "--id=0"],
        capture_output=True, text=True,
    )
    # MIG device UUIDs are listed by `nvidia-smi -L`; parse those lines:
    listing = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
    return [
        line.split("UUID:")[1].strip().rstrip(")")
        for line in listing.splitlines()
        if "MIG" in line and "UUID:" in line
    ]

devices = list_mig_devices()
if devices:
    print("MIG devices found:")
    for d in devices:
        print("  ", d)
    print(f"\nRun a job pinned to the first slice:")
    print(f"  CUDA_VISIBLE_DEVICES={devices[0]} python train.py")
else:
    print("No MIG devices here. On a MIG-enabled GPU, `nvidia-smi -L` exposes")
    print("UUIDs like 'MIG-9a2c...'; pin a process with:")
    print("  CUDA_VISIBLE_DEVICES=MIG-9a2c... python train.py")
    print("\nWith PyTorch inside that process, torch.cuda.device_count() == 1")
    print("and that single device IS the isolated MIG slice.")

## Use Cases

#### Use Case 1: Multi-tenant inference / notebook serving

- **Context**: An 8×A100 node serves dozens of small models and interactive
  Jupyter users; a full A100 per pod wastes 80%+ of each GPU.
- **Implementation**: Enable MIG and create seven `1g.5gb` (A100-40GB) instances
  per GPU → 56 isolated slices per node. Pods request `nvidia.com/mig-1g.5gb: 1`.
- **Results**: Far higher utilization with **hard isolation** — one tenant's OOM
  or fault cannot affect another.

#### Use Case 2: Right-sized training + inference mix

- **Context**: A few medium fine-tuning jobs alongside many small inference
  endpoints on the same node.
- **Implementation**: A mixed geometry per GPU, e.g. `3g.20gb` for the trainer
  plus `2g.10gb` + `2g.10gb` for inference (3+2+2 = 7 SM slices).
- **Results**: Each workload gets a guaranteed compute/memory budget; no
  cross-interference on latency.

#### Use Case 3: CI / ephemeral GPU jobs

- **Context**: Many short test jobs each need *a* GPU, not a fast one.
- **Implementation**: Smallest profile (`1g.5gb`) to maximize concurrency; each
  CI runner pins to one MIG UUID.
- **Results**: High job concurrency, cheap and isolated runners, no GPU
  contention flakiness.

## Best Practices

1. **Match the profile to the workload's real footprint** — both compute *and*
   memory. A `1g.5gb` slice has a hard 5 GB cap; an OOM there is isolated but
   still fatal to that job. Size to the largest batch/model + KV-cache a tenant
   actually needs.
2. **Validate geometry before creating** with `nvidia-smi mig -lgip` and
   `-lgipp` (placements). Not every mix fits the 7 SM / 8 memory slice budget.
3. **Always select MIG devices by UUID**, never integer index — integer CUDA
   device indexing is unsupported across MIG devices.
4. **Free the GPU before mode changes.** Enabling/disabling MIG requires no CUDA
   clients attached and may need a GPU reset; plan a maintenance window.
5. **Destroy in reverse order** — Compute Instances, then GPU Instances, then
   disable MIG mode — to avoid "in use" errors.
6. **On Kubernetes, manage MIG declaratively** (mig-parted / GPU Operator), not
   by hand-running `nvidia-smi mig` on nodes the operator controls — manual edits
   desync the advertised `nvidia.com/mig-*` resources.
7. **Pick a device-plugin strategy deliberately**: `single` (all slices
   identical, advertised as `nvidia.com/gpu`) is simplest; `mixed` (heterogeneous,
   advertised per profile) is more flexible but pods must request exact profiles.

## Common Pitfalls

1. **Expecting MIG on unsupported hardware.** V100/T4 and consumer cards have no
   MIG. *Avoid*: confirm with `nvidia-smi --query-gpu=mig.mode.current --format=csv`
   (it errors / shows `[N/A]` on unsupported GPUs).
2. **Mode change "succeeds" but MIG is still off.** A pending mode change needs a
   GPU reset/reboot. *Avoid*: free the GPU, `nvidia-smi --gpu-reset -i 0`, and
   re-check `mig.mode.current`.
3. **Using integer `CUDA_VISIBLE_DEVICES`.** Apps see no GPU or the wrong slice.
   *Avoid*: always use the `MIG-<UUID>` form.
4. **Invalid geometry.** Asking for more slices than the 7-SM/8-memory budget
   allows (or a layout with no valid placement) makes creation fail. *Avoid*:
   check `-lgip` instance counts and `-lgipp` placements first.
5. **No dynamic resizing of a live instance.** You cannot grow/shrink a GI while
   it's in use; you must destroy and recreate. *Avoid*: plan layouts, and drain
   before reconfiguring.
6. **Forgetting that MIG disables some features.** With MIG on, **NVLink
   peer-to-peer / GPUDirect P2P across instances and full-GPU profiling** are not
   available, and a MIG slice can't span GPUs. *Avoid*: don't MIG GPUs that need
   multi-GPU NVLink collectives.

## Performance Optimization

MIG itself adds no runtime overhead — it *partitions* the silicon. "Optimization"
means choosing a geometry that maximizes useful throughput for your workload mix.

#### Configuration tuning

- **Profile size vs concurrency**: smaller profiles → more instances and higher
  job concurrency, but each has fewer SMs and less bandwidth. Right-size to the
  *largest* job a tenant actually runs, not the smallest that fits.
- **Memory is often the binding constraint**: a `1g.5gb` slice caps at 5 GB.
  Match profile memory to model weights + activations + KV-cache.
- **Don't strand slices**: the SM budget is 7. A `4g.20gb` (4 SMs) leaves only
  room for one `3g` *or* one `2g`+one `1g`; mixing carelessly wastes compute
  slices. Use `-lgipp` to avoid layouts with no valid placement.
- **Bandwidth isolation is the whole point**: unlike time-slicing/MPS, MIG
  instances don't contend for memory bandwidth. Favor MIG when **tail latency**
  and **noisy-neighbor** effects matter more than peak single-job speed.
- **Batch within a slice**: since a slice is fixed-size, raise utilization by
  batching/concurrent streams *inside* the instance rather than expecting the
  hardware to lend it spare capacity (it won't — that's the guarantee).

In [ ]:
# Inspect the live MIG layout + per-instance utilization on a node.
import shutil, subprocess

def show_mig_layout():
    if not shutil.which("nvidia-smi"):
        print("nvidia-smi not present here. On a MIG-enabled GPU node run:")
        print("  nvidia-smi -L                      # MIG devices + UUIDs")
        print("  nvidia-smi mig -lgi                # GPU Instances")
        print("  nvidia-smi mig -lci                # Compute Instances")
        print("  nvidia-smi                         # per-MIG utilization table")
        print("  dcgmi dmon -e 1002,1003            # DCGM per-instance SM/mem activity")
        return
    for args in (["nvidia-smi", "-L"], ["nvidia-smi", "mig", "-lgi"]):
        print("$", " ".join(args))
        try:
            r = subprocess.run(args, capture_output=True, text=True, timeout=20)
            print(r.stdout or r.stderr)
        except Exception as e:
            print("  (could not run:", e, ")")

show_mig_layout()

# NOTE: legacy `nvidia-smi` per-process utilization is limited under MIG; use
# DCGM / dcgm-exporter for accurate PER-INSTANCE metrics
# (e.g. DCGM_FI_PROF_GR_ENGINE_ACTIVE reported per MIG device).

## Production Deployment

In production you rarely run `nvidia-smi mig` by hand. On Kubernetes the **NVIDIA
GPU Operator** + **device plugin** + **GPU Feature Discovery** enable MIG, create
the partitions, and advertise them to the scheduler.

#### Enable MIG via the GPU Operator

```bash
# 'single' = every MIG device identical, advertised as nvidia.com/gpu
# 'mixed'  = heterogeneous profiles, advertised as nvidia.com/mig-<profile>
helm install --wait gpu-operator nvidia/gpu-operator \
  -n gpu-operator --create-namespace \
  --set mig.strategy=mixed

# Choose a per-node geometry (driven by the MIG Manager):
kubectl label node gpu-node-1 nvidia.com/mig.config=all-1g.5gb --overwrite
```

#### Request a MIG slice from a workload (mixed strategy)

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: mig-inference
spec:
  containers:
    - name: app
      image: nvcr.io/nvidia/pytorch:24.05-py3
      command: ["python", "serve.py"]
      resources:
        limits:
          nvidia.com/mig-1g.5gb: 1     # one isolated MIG instance
```

#### Single strategy (homogeneous slices)

```yaml
resources:
  limits:
    nvidia.com/gpu: 1    # the scheduler hands out one 1g.5gb MIG device
```

The container sees exactly one CUDA device — its dedicated MIG slice — with no
code changes required.

## Monitoring and Observability

#### Key signals to track

- **MIG mode & geometry per GPU** — `nvidia-smi --query-gpu=mig.mode.current
  --format=csv` and `nvidia-smi mig -lgi/-lci`. Watch for drift between intended
  and actual layout.
- **Per-instance utilization** — **DCGM / dcgm-exporter** reports metrics *per
  MIG instance* (e.g. `DCGM_FI_PROF_GR_ENGINE_ACTIVE`,
  `DCGM_FI_PROF_DRAM_ACTIVE`), so you can see which slices are busy vs stranded.
  Legacy per-process `nvidia-smi` utilization is limited under MIG — prefer DCGM.
- **Per-instance memory** — `DCGM_FI_DEV_FB_USED` / `FB_FREE` per MIG device to
  catch slices running near their hard cap.
- **Schedulable MIG resources (k8s)** — `kubectl describe node` should show
  `nvidia.com/mig-<profile>` (mixed) or updated `nvidia.com/gpu` (single)
  capacity after partitioning.

```bash
# Per-instance activity stream:
dcgmi dmon -e 1002,1003,1005          # SM active, SM occupancy, mem active

# What slices Kubernetes is advertising:
kubectl describe node gpu-node-1 | grep -i nvidia.com/mig
```

#### Logging best practices

- Tag metrics/logs with the **MIG UUID and profile** so per-tenant dashboards map
  cleanly to physical slices.
- Alert when a slice sits at its memory cap (impending OOM) or at ~0 utilization
  (stranded capacity to reclaim).

## Troubleshooting

#### Issue 1: MIG mode won't enable (stays "Disabled")

**Symptoms**: `nvidia-smi -i 0 -mig 1` reports a *pending* change; `mig.mode.current`
stays `Disabled`.

**Cause**: A CUDA client still holds the GPU, or the mode change needs a GPU
reset/reboot.

**Solution**: Stop all GPU processes (`fuser -k /dev/nvidia*`), then
`sudo nvidia-smi --gpu-reset -i 0` (or reboot the node). Verify with
`nvidia-smi --query-gpu=mig.mode.current --format=csv`.

#### Issue 2: "Unable to create a GPU instance" / insufficient resources

**Symptoms**: `nvidia-smi mig -cgi ...` fails with an availability/placement error.

**Cause**: The requested geometry exceeds the 7-SM / 8-memory-slice budget, or no
valid placement exists for that mix.

**Solution**: Check remaining capacity with `nvidia-smi mig -lgip` (free instance
counts) and `-lgipp` (placements); destroy existing GIs (`-dgi`) and recreate a
valid layout.

#### Issue 3: Application sees no GPU under MIG

**Symptoms**: CUDA reports zero devices, or the wrong slice, inside a container.

**Cause**: `CUDA_VISIBLE_DEVICES` set to an integer index (unsupported with MIG),
or the container wasn't granted a MIG device.

**Solution**: Use the `MIG-<UUID>` form from `nvidia-smi -L`; in Kubernetes
request `nvidia.com/mig-<profile>: 1` and confirm the device plugin advertises it
(`kubectl describe node`).

## Comparison with Alternatives

How MIG compares with the other GPU-sharing mechanisms:

| Aspect | MIG | Time-Slicing | MPS (Multi-Process Service) |
|---|---|---|---|
| Isolation | **Hard** — dedicated SMs, L2, DRAM & bandwidth; fault-isolated | None — processes time-share, no memory isolation | Soft — shared context, optional memory % limits |
| Hardware | A100/A30/H100/H200/B200 only | Any NVIDIA GPU | Any NVIDIA GPU (Volta+) |
| Memory-bandwidth isolation | Yes | No | No |
| Fault isolation | Yes (per instance) | No | No (shared context) |
| Granularity | Fixed profiles (1g/2g/3g/4g/7g) | Arbitrary replica count (oversubscribe) | Arbitrary processes, optional % cap |
| Reconfigure cost | Disruptive: idle GPU + possible reset/reboot | Free, instant (just replicas) | Cheap, per-node daemon |
| Best for | Multi-tenant isolation, predictable QoS | Bursty/low-utilization sharing, dev | Many cooperative processes, one trusted tenant |

### When to choose MIG

Choose **MIG** when:

- You need **hard isolation and QoS guarantees** between tenants on one GPU.
- You have **MIG-capable hardware** and many small-to-medium workloads.
- Predictable tail latency matters more than peak single-job throughput.

Prefer **time-slicing** for cheap oversubscription on any GPU when isolation
isn't needed; prefer **MPS** when one trusted workload has many cooperating
processes. These compose — you can even **time-slice or run MPS *within* a MIG
instance** for a second level of sharing.

## Resources

### Official documentation

- NVIDIA MIG User Guide: <https://docs.nvidia.com/datacenter/tesla/mig-user-guide/>
- MIG product page / overview: <https://www.nvidia.com/en-us/technologies/multi-instance-gpu/>
- GPU Operator — MIG support: <https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-operator-mig.html>
- `nvidia-smi` MIG commands reference (in the MIG User Guide, "MIG Device Management").

### Tutorials and guides

- "Getting the Most Out of the A100 GPU with Multi-Instance GPU" (NVIDIA blog):
  <https://developer.nvidia.com/blog/getting-the-most-out-of-the-a100-gpu-with-multi-instance-gpu/>
- k8s-device-plugin MIG strategies (`single` vs `mixed`):
  <https://github.com/NVIDIA/k8s-device-plugin#configuration-option-details>
- `nvidia-mig-parted` examples (declarative configs):
  <https://github.com/NVIDIA/mig-parted/tree/main/examples>

### Community resources

- NVIDIA Developer Forums (GPU virtualization / MIG):
  <https://forums.developer.nvidia.com/>
- GPU Operator issues / discussions:
  <https://github.com/NVIDIA/gpu-operator/issues>
- Stack Overflow tag: <https://stackoverflow.com/questions/tagged/nvidia-mig>

### Related technologies

- **MIG Manager (`nvidia-mig-parted` / `k8s-mig-manager`)** — declarative,
  fleet-wide MIG configuration.
- **NVIDIA GPU Operator** — enables MIG and wires up the device plugin/GFD.
- **k8s-device-plugin & GPU Feature Discovery** — advertise MIG resources/labels.
- **DCGM / dcgm-exporter** — per-MIG-instance metrics.
- **Time-Slicing & MPS** — alternative GPU-sharing mechanisms.